# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will demonstrate how to load the data via its Croissant schema, review record sets and field `@id`s, extract and process tabular data, and visualize it for analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the FAIR^2 Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and their corresponding field and column `@id`s from the Croissant schema. This will help you understand the data's structure and what is available for extraction.

In [ ]:
# List all RecordSet @id's and meta information
from mlcroissant._src.structure.graph.dataset import RecordSet as MLC_RecordSet

record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets in the schema:\n")
record_set_ids = []
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}\n    @id      : {rs.id}\n    Description: {rs.description}\n    Fields: {[f'@id: {f.id}, name: {f.name}' for f in rs.fields]}\n    Columns: {[f'@id: {c.id}, name: {c.name}' for c in rs.columns]}\n")
    record_set_ids.append(rs.id)

if not record_set_ids:
    raise ValueError("No record sets found in Croissant schema.")

## 3. Data Extraction
Load data from each discovered record set. We use record set and field `@id`s as referenced above. Each record set is loaded into a DataFrame for analysis, and you can explore the column names and first records for each set.

In [ ]:
# Load and preview all available record sets
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set_id}")
    rows = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(rows)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(f"Sample:\n{df.head()}\n")
# Select the first record set for downstream analysis
main_record_set_id = record_set_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
We'll apply common EDA steps: filtering records, normalizing numeric fields, and grouping data. First, let's inspect numeric columns, then filter on one, normalize it, and aggregate by another key attribute.


In [ ]:
# Identify numeric columns in the main DataFrame
numeric_columns = main_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if not numeric_columns:
    print("No numeric columns found in this record set. EDA is not possible!")
else:
    print(f"Numeric columns found: {numeric_columns}")
    # Select the first numeric field for demonstration
    numeric_field_id = numeric_columns[0]
    # Apply threshold filtering for demonstration
    threshold = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in '{main_record_set_id}' with '{numeric_field_id}' > mean:")
    print(filtered_df.head())
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a non-numeric field if available
    group_candidates = [col for col in main_df.columns if main_df[col].dtype == object]
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No suitable non-numeric field for grouping available.")

## 5. Visualization
Visualize the distribution of the normalized numeric field and, if applicable, its grouping by the selected categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_columns:
    print('No numeric columns to visualize.')
else:
    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    # Histogram of selected numeric field
    sns.histplot(main_df[numeric_field_id].dropna(), ax=axs[0], kde=True, bins=30, color='cornflowerblue')
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    axs[0].set_xlabel(numeric_field_id)
    # Histogram of normalized filtered data
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), ax=axs[1], kde=True, bins=30, color='forestgreen')
    axs[1].set_title(f"Normalized {numeric_field_id} (filtered)")
    axs[1].set_xlabel(f"{numeric_field_id}_normalized")
    plt.tight_layout()
    plt.show()
    # If group_field_id exists, plot grouped means
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        plot_df = grouped_df.dropna().sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(data=plot_df, x=numeric_field_id, y=group_field_id, palette='mako')
        plt.title(f"Top 20 '{group_field_id}' by mean '{numeric_field_id}' (filtered)")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
We loaded and explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, extracted record sets by their `@id`, inspected the tabular data, and demonstrated basic filtering, normalization, grouping, and visualizations. For further analysis, you can explore other record sets, fields, or apply custom domain-specific processing to your needs.